# Final Model Selection

Selection uses **training-split evidence only**: the lowest repeated 5x3 CV RMSE among tuned models (`reports/tuning/repeated_cv_top_tuned.csv`, from `scripts/robustness_check.py`), falling back to the lowest 5-fold CV RMSE on the leaderboard. The winner is then evaluated once on the held-out test set and saved to `models/final_model.pkl` + `models/preprocessor.pkl`. The test set never drives the choice.

In [1]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings('ignore')
import pandas as pd
from src.preprocessing import prepare_data
from src.models import make_model, display_name
from src.tuning import build_tuned_model, load_tuned_params
from src.evaluation import evaluate, build_leaderboard, results_table, print_summary
from src.prediction import save_final
from src import config

In [2]:
data = prepare_data()
n_rows = sum(data['meta']['rows'][s] for s in ('train','val','test'))

### Choose the winner (training-split evidence only)

In [3]:
rep_path = config.REPORTS/'tuning'/'repeated_cv_top_tuned.csv'
board = build_leaderboard()
if rep_path.exists():
    rep = pd.read_csv(rep_path).sort_values('rep_cv_rmse_5x3')
    best_key, rule = rep.iloc[0]['key'], 'lowest repeated 5x3 CV RMSE'
    display(rep)
else:
    cand = board[~board['key'].str.startswith('baseline')]
    best_key, rule = cand.iloc[0]['key'], 'lowest 5-fold CV RMSE'
base = best_key.removesuffix('_tuned'); is_tuned = best_key.endswith('_tuned')
print(f'Selected ({rule}):', display_name(base), '(tuned)' if is_tuned else '(default)')

,key,model,cv5_rmse,rep_cv_rmse_5x3,rep_cv_sd,val_rmse,test_rmse,test_mae,test_r2,test_mape
0,xgboost_model_tuned,XGBoost Regressor (tuned),26516.949222,27024.881864,6650.464464,44800.918562,23517.097780,9628.810547,0.808717,0.292907
1,gradient_boosting_tuned,Gradient Boosting Regressor (tuned),26254.254911,27067.248481,6924.657762,44562.517240,24387.174905,9719.830527,0.794301,0.292133
2,lightgbm_model_tuned,LightGBM Regressor (tuned),26541.400904,27432.167900,6286.371555,46065.221812,23725.937413,9775.248487,0.805305,0.296132
3,catboost_model_tuned,CatBoost Regressor (tuned),26984.936206,27869.524529,7054.594532,45749.166890,23300.137786,9525.939192,0.812230,0.296483
4,random_forest_tuned,Random Forest Regressor (tuned),28128.320990,28624.050816,6741.638557,44710.179236,21797.711757,9145.529843,0.835665,0.293254
5,lasso_regression_tuned,Lasso Regression (tuned),29644.271405,29716.696511,7358.028349,52930.017176,28714.755658,10560.862254,0.714820,0.315304


Selected (lowest repeated 5x3 CV RMSE): XGBoost Regressor (tuned)


### Refit on the training split and evaluate once on the test set

In [4]:
params = load_tuned_params().get(base, {}) if is_tuned else {}
model = build_tuned_model(base, data['target_transform'], params) if is_tuned else make_model(base, data['target_transform'])
print('Hyperparameters:', params or 'library defaults')
results, pred, _ = evaluate(model, data, display_name(base) + (' (tuned)' if is_tuned else ''), best_key, evaluate_test=True)
print_summary(results, n_rows, len(data['feature_names']))
results_table(results)

Hyperparameters: {'colsample_bytree': 0.8776807051588262, 'gamma': 0.001, 'learning_rate': 0.04749239763680407, 'max_depth': 3, 'min_child_weight': 4.059382607081596, 'n_estimators': 676, 'reg_alpha': 0.755681014127442, 'reg_lambda': 2.1154290797261215, 'subsample': 0.9757995766256756}


MODEL: XGBoost Regressor (tuned)
Dataset:  Rows: 4735  Features: 34   Fingerprint: OK (70560d1bd70af7cd)
Train RMSE: 20,365.58   Validation RMSE: 44,800.92   Test RMSE: 23,517.10
Validation MAE: 11,958.03   Validation MSE: 2,007,122,304.00   Validation R2: 0.6175
Test MAE: 9,628.81   Test MSE: 553,053,888.00   Test R2: 0.8087
CV RMSE: 26,516.95 (+/- 7,085.33, 5-fold on train)
Training Time: 0.30 s   (CV time: 1.40 s)


,MAE,MSE,RMSE,R2,Adj_R2,MAPE,CV RMSE,Train time (s)
train,7410.303223,4.147567e+08,20365.577232,0.876385,0.875103,0.224044,26516.949222,0.296737
val,11958.032227,2.007122e+09,44800.918562,0.617473,0.598233,0.291008,26516.949222,0.296737
test,9628.810547,5.530539e+08,23517.097780,0.808717,0.799096,0.292907,26516.949222,0.296737


In [5]:
meta = {'model': display_name(base) + (' (tuned)' if is_tuned else ''), 'key': best_key,
        'selection_rule': rule, 'hyperparameters': params,
        'reference_id': data['fingerprint']['reference']['reference_id'],
        'cv_rmse': results['cv']['RMSE_mean'], 'val_rmse': results['val']['RMSE'],
        'test_rmse': results['test']['RMSE'], 'test_mae': results['test']['MAE'],
        'test_r2': results['test']['R2'], 'test_mape': results['test']['MAPE'],
        'target_transform': data['target_transform']}
save_final(model, data['preprocessor'], data['feature_names'], meta)
print('Saved models/final_model.pkl + models/preprocessor.pkl'); meta

Saved models/final_model.pkl + models/preprocessor.pkl


{'model': 'XGBoost Regressor (tuned)',
 'key': 'xgboost_model_tuned',
 'selection_rule': 'lowest repeated 5x3 CV RMSE',
 'hyperparameters': {'colsample_bytree': 0.8776807051588262,
  'gamma': 0.001,
  'learning_rate': 0.04749239763680407,
  'max_depth': 3,
  'min_child_weight': 4.059382607081596,
  'n_estimators': 676,
  'reg_alpha': 0.755681014127442,
  'reg_lambda': 2.1154290797261215,
  'subsample': 0.9757995766256756},
 'reference_id': '70560d1bd70af7cd',
 'cv_rmse': 26516.949222091316,
 'val_rmse': 44800.91856201165,
 'test_rmse': 23517.097780125845,
 'test_mae': 9628.810546875,
 'test_r2': 0.8087170124053955,
 'test_mape': 0.2929067611694336,
 'target_transform': 'log1p'}